# Dhaka PM2.5 — Part 1: Get the data ready

We are building an early-warning system for air pollution in Dhaka.
Given the last few hours of PM2.5 readings, predict how bad the air will be next.

**This notebook does one job: load the data and clean it.**
No features yet. No model yet. Those come in Part 2 and Part 3.

Run the cells top to bottom. Every code cell has a description above it
explaining what it does and why.

> Use a **CPU** runtime (Runtime -> Change runtime type -> CPU).
> Spark's machine learning library only runs on CPU, so a GPU does nothing here.

## Step 1 — See what Python and Java we have

Spark is written in Java. PySpark is only a Python wrapper that talks to a Java
program running in the background. So we need **both** Python and Java.

This cell just prints the versions. Nothing is installed or changed yet.

In [ ]:
import sys
import subprocess

print("Python version:", sys.version.split()[0])

# Java prints its version to stderr, not stdout, which is why we read .stderr
java = subprocess.run(["java", "-version"], capture_output=True, text=True)
print("Java version  :", java.stderr.strip().splitlines()[0] if java.stderr else "NOT FOUND")

## Step 2 — Install PySpark

`pip install` downloads about 300 MB, so we skip it if PySpark is already here.

We ask for version **4.0.4** on purpose:

- Colab now runs Python 3.13, and older PySpark 3.5 does not support it.
- Colab preinstalls a package that expects PySpark 4.0.x, so this version keeps
  pip from printing a conflict warning.

Takes 1-2 minutes the first time.

In [ ]:
try:
    import pyspark
    print("PySpark already installed:", pyspark.__version__)
except ImportError:
    print("Installing PySpark, please wait...")
    !pip install -q pyspark==4.0.4
    import pyspark
    print("Installed PySpark:", pyspark.__version__)

## Step 3 — Download the data

The readings come from the **US Embassy in Dhaka**, which has measured PM2.5
every hour since 2016 and publishes it through the AirNow programme.

One CSV file per year, ten files, about 8 MB in total.

Many tutorials point at `dosairnowdata.org` — that site no longer exists.
The address below is the live one, the same storage the EPA's own embassy map
reads from. It is public, so no login or API key is needed.

The `if` check means re-running this cell costs nothing.

In [ ]:
import os
import urllib.request

BASE_URL = ("https://s3-us-west-1.amazonaws.com/files.airnowtech.org"
            "/airnow/EmbassyHistorical/Dhaka")
FOLDER = "data"

os.makedirs(FOLDER, exist_ok=True)

for year in range(2016, 2026):
    filename = f"Dhaka_PM2.5_{year}_YTD.csv"
    filepath = os.path.join(FOLDER, filename)

    if os.path.exists(filepath):
        print("already downloaded:", filename)
    else:
        urllib.request.urlretrieve(f"{BASE_URL}/{year}/{filename}", filepath)
        print("downloaded       :", filename)

print()
print("files in", FOLDER, "->", len(os.listdir(FOLDER)))

## Step 4 — Peek at the raw file with plain Python

Before starting Spark, look at the actual text. Knowing the exact column names
and how the date is written saves a lot of guessing later.

Notice two things in the output:

- Some column names contain **spaces and dots** — `Raw Conc.`, `QC Name`.
  Those are annoying to type in Spark, so we will rename them in Step 6.
- The very first data row has `-999.0` and `Missing`. That is the code the
  sensor uses for *no reading*. Not a real measurement. Step 9 removes them.

In [ ]:
with open("data/Dhaka_PM2.5_2016_YTD.csv") as f:
    for i in range(4):
        print(f.readline().strip())
        print()

## Step 5 — Start Spark

A `SparkSession` is the doorway to everything in Spark. One per notebook.

What the settings mean:

| Setting | Meaning |
|---|---|
| `master("local[*]")` | Run on this machine, using all its CPU cores |
| `appName(...)` | A name, only shows up in logs |
| `timeZone("Asia/Dhaka")` | The readings are Dhaka local time, so Spark should agree |

The first run takes ~20 seconds because a Java process has to start up.

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("dhaka-pm25")
         .master("local[*]")
         .config("spark.sql.session.timeZone", "Asia/Dhaka")
         .getOrCreate())

# Spark prints a lot of INFO/WARN noise by default. Show only real errors.
spark.sparkContext.setLogLevel("ERROR")

print("Spark is running, version", spark.version)

## Step 6 — Describe the columns to Spark

Spark can guess column types by reading the whole file first (`inferSchema`).
We do **not** do that, for two reasons:

1. Guessing means reading all the data twice — slow.
2. Part 3 uses streaming, and Spark **refuses** to guess for a stream.
   So we have to write this out sooner or later. Better now, once.

Bonus: because we supply the names ourselves, Spark ignores the messy header
in the file and uses our clean names instead. No renaming step needed.

`StringType` is text, `IntegerType` is a whole number, `DoubleType` is a decimal.

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType,
)

# The order here MUST match the column order in the CSV file.
schema = StructType([
    StructField("site",         StringType()),    # always "Dhaka"
    StructField("parameter",    StringType()),    # always "PM2.5 - Principal"
    StructField("date_text",    StringType()),    # e.g. "2016-01-01 01:00 AM"
    StructField("year",         IntegerType()),
    StructField("month",        IntegerType()),
    StructField("day",          IntegerType()),
    StructField("hour",         IntegerType()),   # 0 to 23
    StructField("nowcast",      DoubleType()),    # AirNow's smoothed value
    StructField("aqi",          IntegerType()),   # Air Quality Index, 0-500
    StructField("aqi_category", StringType()),    # "Good" ... "Hazardous"
    StructField("pm25",         DoubleType()),    # <-- THE VALUE WE PREDICT
    StructField("unit",         StringType()),    # always "UG/M3"
    StructField("duration",     StringType()),    # always "1 Hr"
    StructField("qc",           StringType()),    # "Valid" / "Missing" / ...
])

print("schema has", len(schema.fields), "columns")

## Step 7 — Read all ten CSV files at once

Give Spark the **folder**, not a filename, and it reads every CSV inside as one
big table.

`header=True` tells Spark the first line of each file is a header row to skip —
we already supplied our own names in Step 6.

In [ ]:
# .cache() keeps the table in memory after the first read, so the many
# .count() and .show() calls below do not re-read the files each time.
raw = spark.read.option("header", True).schema(schema).csv("data").cache()

print("rows:", raw.count())
raw.show(5)

## Step 8 — Confirm the boring columns really are boring

Four columns look like they never change. If that is true they carry zero
information and can be dropped.

Never assume this — check. `distinct()` lists the different values a column
actually holds.

In [ ]:
for column in ["site", "parameter", "unit", "duration"]:
    values = [row[0] for row in raw.select(column).distinct().collect()]
    print(f"{column:<10} -> {values}")

## Step 9 — Look at the quality flag

Every row carries a `qc` flag from the monitoring station.

You will see that most rows are `Valid`, but a few thousand are **not**. Those
rows hold `-999.0` instead of a real reading.

This matters a lot. `-999` is not a small error — it is roughly ten times
further from a normal reading than the worst real pollution ever recorded. Train
on those and the model learns nonsense.

In [ ]:
raw.groupBy("qc").count().orderBy("count", ascending=False).show()

## Step 10 — Keep only the good rows

`filter` keeps the rows where the condition is true.

We check the sentinel value **as well as** the flag, belt and braces: a handful
of rows are flagged `Valid` but still carry `-999` in the `nowcast` column.

In [ ]:
from pyspark.sql import functions as F

good = raw.filter(F.col("qc") == "Valid").filter(F.col("pm25") != -999.0)

print("rows before:", raw.count())
print("rows after :", good.count())
print("removed    :", raw.count() - good.count())

## Step 11 — Build a real timestamp

Right now the time is spread across four separate number columns, plus a text
column written in 12-hour format (`01:00 AM`).

We want **one** proper timestamp column, because sorting by time is the whole
basis of this project — every feature we build later is "what was the reading
N hours ago".

We build it from `year`/`month`/`day`/`hour` rather than parsing the text.
Both give the same answer here, but reading four numbers can never trip over
AM/PM or date-format differences.

In [ ]:
good = good.withColumn(
    "ts", F.expr("make_timestamp(`year`, `month`, `day`, `hour`, 0, 0)"))

good.select("date_text", "year", "month", "day", "hour", "ts").show(5, truncate=False)

## Step 12 — Fix impossible values

The lowest reading in the file is **-4.0**. A negative amount of dust in the air
cannot exist — it is sensor noise wobbling around zero on very clean hours.

`greatest(pm25, 0.0)` returns whichever is bigger, so anything negative becomes
0 and everything else is untouched. We nudge rather than delete, because
throwing the row away would punch a hole in the hourly sequence.

In [ ]:
print("before -> lowest reading:", good.agg(F.min("pm25")).collect()[0][0])

good = good.withColumn("pm25", F.greatest(F.col("pm25"), F.lit(0.0)))

print("after  -> lowest reading:", good.agg(F.min("pm25")).collect()[0][0])

## Step 13 — Drop the columns we no longer need

- `site`, `parameter`, `unit`, `duration` — proven constant in Step 8
- `date_text`, `year`, `month`, `day` — replaced by `ts` in Step 11
- `qc` — every remaining row is `Valid`, so the column says nothing now

We keep `hour` because time of day genuinely affects pollution: traffic peaks in
the morning and evening.

In [ ]:
clean = good.drop("site", "parameter", "unit", "duration",
                  "date_text", "year", "month", "day", "qc").cache()

clean.printSchema()
clean.show(5)

## Step 14 — Get to know the data

`describe()` gives count, average, spread, smallest and largest.

Two things worth noticing in the output:

- The average is very high. For reference, the World Health Organization
  guideline for a 24-hour average is **15 ug/m3**.
- The largest value is enormous compared to the average, so the data has rare
  extreme spikes. Those spikes are exactly what an early-warning system exists
  to catch.

In [ ]:
clean.select("pm25").describe().show()

### How often is the air in each category?

`groupBy` collects rows that share a value, then `count` says how many landed in
each group. This is the government's own health scale.

In [ ]:
(clean.groupBy("aqi_category")
      .count()
      .orderBy("count", ascending=False)
      .show(truncate=False))

## Step 15 — How many hours are missing?

The station did not record every single hour. Sometimes it was offline for days.

Counting the gaps now matters, because in Part 2 we build features like
"the reading one hour ago". If an hour is missing, "one row back" is **not**
"one hour back", and the model would silently learn from the wrong number.

Below we compare how many hours *should* exist between the first and last
reading against how many rows we actually have.

In [ ]:
first_ts, last_ts = clean.agg(F.min("ts"), F.max("ts")).collect()[0]

# 3600 seconds in an hour. +1 because both ends are included.
hours_expected = int((last_ts - first_ts).total_seconds() / 3600) + 1
hours_we_have = clean.count()

print("first reading  :", first_ts)
print("last reading   :", last_ts)
print("hours expected :", f"{hours_expected:,}")
print("hours we have  :", f"{hours_we_have:,}")
print("hours MISSING  :", f"{hours_expected - hours_we_have:,}",
      f"({(hours_expected - hours_we_have) / hours_expected:.1%})")

## Step 16 — Draw one month

Numbers only tell you so much. Plotting shows the daily rhythm and the spikes.

`toPandas()` pulls Spark data into ordinary Python so matplotlib can draw it.
Only ever do this on a **small** slice — it loads everything into memory at once.
Here we take a single month.

In [ ]:
import matplotlib.pyplot as plt

one_month = (clean
             .filter((F.col("ts") >= "2024-01-01") & (F.col("ts") < "2024-02-01"))
             .orderBy("ts")
             .toPandas())

plt.figure(figsize=(14, 4))
plt.plot(one_month["ts"], one_month["pm25"], linewidth=1)
plt.axhline(15, color="green", linestyle="--", label="WHO 24h guideline (15)")
plt.ylabel("PM2.5 (ug/m3)")
plt.title("Dhaka PM2.5, hourly, January 2024")
plt.legend()
plt.tight_layout()
plt.show()

## Done. What we have now

A `clean` table with one row per recorded hour, holding:

| Column | Meaning |
|---|---|
| `ts` | the timestamp |
| `hour` | hour of day, 0-23 |
| `pm25` | the pollution reading — **the number we will predict** |
| `nowcast` | AirNow's smoothed version of the reading |
| `aqi`, `aqi_category` | the official index and its health label |

### Coming in Part 2

1. Fill in the missing hours with blanks, so "one row back" always means
   "one hour back"
2. Build the inputs: the reading 1 hour ago, 2 hours ago, 24 hours ago, and
   rolling averages
3. Split by **time** — train on older data, test on newer

### Coming in Part 3

Train the model, and check it beats the dumbest possible forecast:
*"the next hour will be the same as this hour."* If it cannot beat that, it has
learned nothing.